# Stage 11 — Kafka Offsets & Replay

**Goal:** לדעת למה Checkpoint ו־Sink צריכים להשתנות אטומית.

**Setup:** Python 3.11+, Jupyter/VS Code; local synthetic examples only.

**CS bridge:** Explain the invariant and its failure mode before running code.

**Source:** https://kafka.apache.org/documentation/

**Next:** Complete the graded lab under ../labs/ and pass an unseen transfer challenge.

## 1. Predict first

לפני Run All: כתוב מה אתה מצפה שה־assert יוכיח ומה אינו בודק. לאחר מכן הרץ את התא.

In [ ]:
import sqlite3
db=sqlite3.connect(":memory:")
db.executescript("""
create table events(id text primary key,value integer);
create table checkpoints(partition_id integer primary key,next_offset integer);
""")
db.execute("begin")
db.execute("insert into events values('A',10)")
db.execute("insert into checkpoints values(0,1)")
db.commit()
assert db.execute("select next_offset from checkpoints").fetchone()[0]==1

## 2. Failure mode / independent check

שנה פרמטר אחד בענף האישי והסבר מדוע הקוד נכשל או משנה תוצאה. החזר את הערך המקורי.

In [ ]:
db.execute("begin")
db.execute("insert into events values('B',20)")
db.execute("update checkpoints set next_offset=2 where partition_id=0")
db.rollback()
assert db.execute("select count(*) from events").fetchone()[0]==1
assert db.execute("select next_offset from checkpoints").fetchone()[0]==1
print("Crash before commit rolls back both sink and offset checkpoint.")
db.close()

## 3. Transfer (no AI)

מה קורה אם SQLite עשה Commit אבל Consumer לא התחייב על Offset ב־Kafka? הסבר At-least-once.

אל תסמן את השלב כהושלם על בסיס מחברת זו בלבד; חובה לבצע את המעבדה ובדיקת המנטור.